# Hostile Testing Phase 6 - Geo & Census Modules

## Purpose
Test geocoding and census data functions with hostile inputs

## Goal
Add 10+ functions to reach 6-7% coverage

In [ ]:
import pandas as pd
import tempfile
from pathlib import Path

test_results = {
    'function': [],
    'module': [],
    'test_category': [],
    'passed': [],
    'error_message': [],
    'severity': []
}

def record_test(function, module, test_category, passed, error_message="", severity="medium"):
    test_results['function'].append(function)
    test_results['module'].append(module)
    test_results['test_category'].append(test_category)
    test_results['passed'].append(passed)
    test_results['error_message'].append(error_message)
    test_results['severity'].append(severity)

def hostile_test(func, test_name, *args, **kwargs):
    try:
        result = func(*args, **kwargs)
        return (True, result, "")
    except Exception as e:
        return (False, None, str(e))

print("Phase 6 test harness loaded")

## Section 1: Geo - Geocoding

In [ ]:
from siege_utilities import concatenate_addresses

# Test concatenate_addresses with hostile address components
hostile_addresses = [
    {"street": "'; DROP TABLE addresses; --", "city": "LA", "state": "CA", "zip": "90001"},
    {"street": "<script>alert(1)</script>", "city": "../../../etc", "state": "XX", "zip": "00000"},
    {"street": "A" * 1000, "city": "B" * 1000, "state": "CC", "zip": "99999"},
    {"street": "123\x00Main", "city": "City\x00Name", "state": "CA", "zip": "90001"},
    {"street": "", "city": "", "state": "", "zip": ""},
    {"street": None, "city": None, "state": None, "zip": None},
    {"street": "123 Main", "city": "Los; rm -rf /", "state": "CA", "zip": "90001"},
    {"street": "数据街", "city": "北京", "state": "BJ", "zip": "100000"},
]

for addr in hostile_addresses:
    success, result, error = hostile_test(
        concatenate_addresses,
        "hostile_addr",
        addr.get('street'),
        addr.get('city'),
        addr.get('state'),
        addr.get('zip')
    )
    record_test(
        "concatenate_addresses",
        "geo.geocoding",
        "hostile_addresses",
        success,
        error,
        "critical" if "DROP TABLE" in str(addr.get('street','')) else "high" if "../" in str(addr.get('city','')) else "medium"
    )

print(f"Completed {len([r for r in test_results['module'] if 'geocoding' in r])} geocoding tests")

## Section 2: Geo - Census Data Selector

In [ ]:
from siege_utilities import (
    get_census_data_selector, select_census_datasets,
    get_analysis_approach, get_dataset_compatibility_matrix
)

# Test get_census_data_selector (no args)
success, result, error = hostile_test(get_census_data_selector, "get_selector")
record_test("get_census_data_selector", "geo.census_data_selector", "basic_call", success, error, "low")
print(f"get_census_data_selector: {'✅' if success else '❌'}")

# Test select_census_datasets with hostile analysis types
hostile_analysis = [
    None, "", "'; DROP TABLE datasets; --", "<script>alert(1)</script>",
    "../../../etc/passwd", "A" * 1000, "analysis\x00type",
    "demographic", "housing", "economic"  # Valid ones
]

for analysis_type in hostile_analysis:
    success, result, error = hostile_test(select_census_datasets, "hostile_analysis", analysis_type)
    record_test(
        "select_census_datasets",
        "geo.census_data_selector",
        "hostile_analysis_types",
        success,
        error,
        "high" if "DROP" in str(analysis_type) else "medium"
    )

# Test get_analysis_approach with hostile inputs
for analysis_type in hostile_analysis[:5]:  # First 5 only
    success, result, error = hostile_test(get_analysis_approach, "hostile_approach", analysis_type)
    record_test(
        "get_analysis_approach",
        "geo.census_data_selector",
        "hostile_analysis_types",
        success,
        error,
        "medium"
    )

print(f"Completed census data selector tests")

## Section 3: Geo - Census Dataset Mapper

In [ ]:
from siege_utilities import (
    get_census_dataset_mapper, list_datasets_by_type, list_datasets_by_geography
)

# Test get_census_dataset_mapper (no args)
success, result, error = hostile_test(get_census_dataset_mapper, "get_mapper")
record_test("get_census_dataset_mapper", "geo.census_dataset_mapper", "basic_call", success, error, "low")
print(f"get_census_dataset_mapper: {'✅' if success else '❌'}")

# Test list_datasets_by_type with hostile types
hostile_types = [
    None, "", "'; DROP TABLE types; --", "<script>alert(1)</script>",
    "../../../etc/passwd", "A" * 1000, "type\x00null",
    "demographic", "housing"  # Valid
]

for dtype in hostile_types:
    success, result, error = hostile_test(list_datasets_by_type, "hostile_type", dtype)
    record_test(
        "list_datasets_by_type",
        "geo.census_dataset_mapper",
        "hostile_types",
        success,
        error,
        "high" if "DROP" in str(dtype) else "medium"
    )

# Test list_datasets_by_geography with hostile geography levels
hostile_geo = [
    None, "", "'; DROP TABLE geography; --", "../../../etc",
    "A" * 1000, "geo\x00null",
    "state", "county", "tract"  # Valid
]

for geo_level in hostile_geo:
    success, result, error = hostile_test(list_datasets_by_geography, "hostile_geo", geo_level)
    record_test(
        "list_datasets_by_geography",
        "geo.census_dataset_mapper",
        "hostile_geography",
        success,
        error,
        "high" if "DROP" in str(geo_level) else "medium"
    )

print(f"Completed census dataset mapper tests")

## Section 4: Geo - Spatial Data (Basic)

In [ ]:
from siege_utilities import get_available_years, discover_boundary_types

# Test get_available_years (no args)
success, result, error = hostile_test(get_available_years, "get_years")
record_test("get_available_years", "geo.spatial_data", "basic_call", success, error, "low")
print(f"get_available_years: {'✅' if success else '❌'}")

# Test discover_boundary_types with hostile years
hostile_years = [
    None, "", "'; DROP TABLE years; --", -1, 0, 9999, 1800, 3000,
    "<script>alert(1)</script>", "../../etc/passwd",
    2020, 2021  # Valid
]

for year in hostile_years:
    success, result, error = hostile_test(discover_boundary_types, "hostile_year", year)
    record_test(
        "discover_boundary_types",
        "geo.spatial_data",
        "hostile_years",
        success,
        error,
        "high" if "DROP" in str(year) else "medium"
    )

print(f"Completed spatial data tests")

## Section 5: Generate Phase 6 Results

In [ ]:
df = pd.DataFrame(test_results)

print("\n" + "="*80)
print("PHASE 6 HOSTILE TESTING SUMMARY")
print("="*80)
print(f"\nTests run: {len(df)}")
print(f"Passed: {df['passed'].sum()}")
print(f"Failed: {(~df['passed']).sum()}")
print(f"Pass rate: {df['passed'].sum() / len(df) * 100:.1f}%")

unique = df['function'].nunique()
print(f"\nUnique functions: {unique}")
print(f"Phase 6 coverage: {unique}/751 = {unique/751*100:.1f}%")

print("\n" + "="*80)
print("RESULTS BY MODULE")
print("="*80)
summary = df.groupby('module').agg({'passed': ['sum', 'count']})
summary.columns = ['passed', 'total']
summary['pass_rate'] = (summary['passed'] / summary['total'] * 100).round(1)
print(summary)

failures = df[~df['passed']]
if len(failures) > 0:
    print("\n" + "="*80)
    print("FAILURES BY SEVERITY")
    print("="*80)
    print(failures.groupby('severity').size())
else:
    print("\n✅ NO FAILURES")

df.to_csv("hostile_testing_phase6_results.csv", index=False)
print(f"\nSaved: hostile_testing_phase6_results.csv")

print("\n" + "="*80)
print("FUNCTIONS TESTED")
print("="*80)
for func in df['function'].unique():
    tests = df[df['function'] == func]
    passed = tests['passed'].sum()
    total = len(tests)
    print(f"{func}: {passed}/{total} ({passed/total*100:.0f}%)")

print("\n" + "="*80)
print("CUMULATIVE COVERAGE (Phases 1-6)")
print("="*80)
total_unique = 38 + unique
print(f"Total unique functions: ~{total_unique}")
print(f"Cumulative coverage: ~{total_unique}/751 = {total_unique/751*100:.1f}%")
print(f"\n🎯 Progress to 25% goal: {total_unique}/188 = {total_unique/188*100:.1f}%")